**Importer les libraries pertinentes**

*Références consultées*  
Řehůřek, R. (2018). *Gensim Doc2Vec Tutorial on the IMDB Sentiment Dataset*. GitHub. https://github.com/piskvorky/gensim/blob/3c3506d51a2caf6b890de3b1b32a8b85f7566ca5/docs/notebooks/doc2vec-IMDB.ipynb

Li, S. (2018). Multi-Class Text Classification with Doc2Vec & Logistic Regression. Medium. https://towardsdatascience.com/multi-class-text-classification-with-doc2vec-logistic-regression-9da9947b43f4

In [129]:
import pandas as pd
from os import listdir
from os import path
import numpy as np
from tqdm import tqdm
tqdm.pandas(desc="progress-bar")
from IPython.display import clear_output
import re
import seaborn as sns
import matplotlib.pyplot as plt

import multiprocessing
cores = multiprocessing.cpu_count()

### NLTK : Stopword filtering & tokenization
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize

### Gensim : Doc2Vec embeddings 
from gensim.models import Doc2Vec
from gensim.models.doc2vec import TaggedDocument

### scikit-learn (Algorithmes de classification)
from sklearn import utils
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.naive_bayes import MultinomialNB # Naive Bayes

**Charger les données**

In [130]:
df = pd.read_excel('../1-data/training_datasets/train_dataset_40pc.xlsx')
df = df[['text_post', 'category']]
df

,text_post,category
0,find a hobby you're interested in that has a l...,incel
1,"Well, you can be realistic about your looks an...",incel
2,Oh yeah. You're the chick who I commented on a...,incel
3,Honestly this is beyond disturbing.,incel
4,Is it really true that americans dislike sharp...,incel
...,...,...
49995,"Okay I’m fairly new to investing, what about s...",neutral
49996,How is it not? \n\nJust because it comes with ...,neutral
49997,Absolutely!,neutral
49998,Mine does this all the time On her piece of wo...,neutral


**Nettoyage**

In [131]:
import string
punct = string.punctuation
punct += '’'

punct

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~’'

In [132]:
def cleanText(text):
    text = re.sub(f'[{punct}]', '', text) 
    text = text.replace('\n', '')
    text = text.lower()
    return text

df['text_post'] = df['text_post'].astype(str).apply(cleanText)
df

,text_post,category
0,find a hobby youre interested in that has a lo...,incel
1,well you can be realistic about your looks and...,incel
2,oh yeah youre the chick who i commented on at ...,incel
3,honestly this is beyond disturbing,incel
4,is it really true that americans dislike sharp...,incel
...,...,...
49995,okay im fairly new to investing what about som...,neutral
49996,how is it not just because it comes with a rel...,neutral
49997,absolutely,neutral
49998,mine does this all the time on her piece of wo...,neutral


**Tokenisation et filtrage**

In [133]:
stopw = stopwords.words('english')
for w in stopw:
    w_nopunct = re.sub(f'[{punct}]', '', w)
    if w_nopunct not in stopw:
        stopw.append(w_nopunct)

stopw

['i',
 'me',
 'my',
 'myself',
 'we',
 'our',
 'ours',
 'ourselves',
 'you',
 "you're",
 "you've",
 "you'll",
 "you'd",
 'your',
 'yours',
 'yourself',
 'yourselves',
 'he',
 'him',
 'his',
 'himself',
 'she',
 "she's",
 'her',
 'hers',
 'herself',
 'it',
 "it's",
 'its',
 'itself',
 'they',
 'them',
 'their',
 'theirs',
 'themselves',
 'what',
 'which',
 'who',
 'whom',
 'this',
 'that',
 "that'll",
 'these',
 'those',
 'am',
 'is',
 'are',
 'was',
 'were',
 'be',
 'been',
 'being',
 'have',
 'has',
 'had',
 'having',
 'do',
 'does',
 'did',
 'doing',
 'a',
 'an',
 'the',
 'and',
 'but',
 'if',
 'or',
 'because',
 'as',
 'until',
 'while',
 'of',
 'at',
 'by',
 'for',
 'with',
 'about',
 'against',
 'between',
 'into',
 'through',
 'during',
 'before',
 'after',
 'above',
 'below',
 'to',
 'from',
 'up',
 'down',
 'in',
 'out',
 'on',
 'off',
 'over',
 'under',
 'again',
 'further',
 'then',
 'once',
 'here',
 'there',
 'when',
 'where',
 'why',
 'how',
 'all',
 'any',
 'both',
 'each

In [134]:
df['tokens'] = df['text_post'].apply(lambda x: word_tokenize(x))

In [135]:
def filter(tokens):
    return [token for token in tokens if len(token) >2 and not token in stopw]

df['tokens'] = df['tokens'].apply(lambda x: filter(x))
df

,text_post,category,tokens
0,find a hobby youre interested in that has a lo...,incel,"[find, hobby, interested, local, club, hang]"
1,well you can be realistic about your looks and...,incel,"[well, realistic, looks, still, confidence, hi..."
2,oh yeah youre the chick who i commented on at ...,incel,"[yeah, chick, commented, rrateme, nothanks, su..."
3,honestly this is beyond disturbing,incel,"[honestly, beyond, disturbing]"
4,is it really true that americans dislike sharp...,incel,"[really, true, americans, dislike, sharpnarrow..."
...,...,...,...
49995,okay im fairly new to investing what about som...,neutral,"[okay, fairly, new, investing, something, like..."
49996,how is it not just because it comes with a rel...,neutral,"[comes, relatively, strict, appraisal, process..."
49997,absolutely,neutral,[absolutely]
49998,mine does this all the time on her piece of wo...,neutral,"[mine, time, piece, wood, already, put, wood, ..."


In [136]:
tagged_documents = df.apply(
    lambda x: TaggedDocument(words=x.tokens, tags=x.category), axis=1)

tagged_documents.values[1]

TaggedDocument(words=['well', 'realistic', 'looks', 'still', 'confidence', 'high', 'school', 'anymore', 'bunch', 'terrible', 'kids', 'cruel', 'anymoreunless', 'still', 'school', 'focus', 'things', 'say', 'confident', 'cut', 'negativity'], tags='incel')

**Vectorisation avec Doc2Vec**

*Distributed Bag of Words (DBOW)* (Continuuous Bag of Words)

In [137]:
model_dbow = Doc2Vec(
    dm=0, 
    vector_size=500, 
    negative=5, 
    hs=0, 
    min_count=2, 
    sample = 0, 
    workers=cores
)

model_dbow.build_vocab([x for x in tqdm(tagged_documents.values)])

100%|██████████| 50000/50000 [00:00<00:00, 2949083.14it/s]


In [138]:
%%time
for epoch in range(30):
    model_dbow.train(utils.shuffle([x for x in tqdm(tagged_documents.values)]), total_examples=len(tagged_documents.values), epochs=1)
    model_dbow.alpha -= 0.002
    model_dbow.min_alpha = model_dbow.alpha

100%|██████████| 50000/50000 [00:00<00:00, 3857044.07it/s]


100%|██████████| 50000/50000 [00:00<00:00, 1205959.78it/s]


CPU times: total: 10min 6s
Wall time: 2min 39s


*Distributed Memory (DM)* (skip-gram)